In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.layers import Dense , Conv2D , GlobalAveragePooling2D , MaxPooling2D
from tensorflow.keras import Sequential

In [2]:
import matplotlib.pyplot as plt
import cv2
import random
import os

In [3]:

# Dataset path
DATASET_PATH = r"C:\Users\ANIKET\Desktop\projects\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"

In [4]:
# Image dimensions
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30



In [5]:
# Data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)



In [6]:
trr = image_dataset_from_directory(
    directory = DATASET_PATH,
    label_mode = 'categorical',
    batch_size = 32,
    image_size = (256 ,  256)

)

classnames = trr.class_names


Found 12446 files belonging to 1 classes.


In [7]:
train = ImageDataGenerator(rescale = 1 / 255.)
test = ImageDataGenerator(rescale = 1 / 255.)
val = ImageDataGenerator(rescale = 1 / 255.)

In [8]:
train_data = train.flow_from_directory(
    directory = DATASET_PATH,
    batch_size = 32,
    target_size = (256, 256),
    class_mode = 'categorical'
)

test_data = train.flow_from_directory(
    directory = DATASET_PATH,
    batch_size = 32,
    target_size = (256, 256),
    class_mode = 'categorical'
)

val_data = train.flow_from_directory(
    directory = DATASET_PATH,
    batch_size = 32,
    target_size = (256, 256),
    class_mode = 'categorical'
)

Found 12446 images belonging to 1 classes.
Found 12446 images belonging to 1 classes.
Found 12446 images belonging to 1 classes.


In [9]:
# Let's make some checkpoints
import datetime

# Early Stopping

def early():
  return tf.keras.callbacks.EarlyStopping(patience = 10 , monitor = 'val_loss' , min_delta =0.001 , restore_best_weights = True)


# Model Checkpoint

def point(path):
  return tf.keras.callbacks.ModelCheckpoint(
        path + ".weights.h5" , 
        monitor = 'val_loss' , 
        save_best_only= True , 
        save_weights_only=True , 
        save_freq = "epoch"
  )

# TensorBoard

def board(dir , name):
  folder = dir + "/" + name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
  return tf.keras.callbacks.TensorBoard(log_dir = folder)




In [10]:
base = tf.keras.applications.Xception(
    include_top=False,
    weights='imagenet')

In [11]:
base.trainable = False

In [12]:
model2 = Sequential()
model2.add(base)
model2.add(GlobalAveragePooling2D())
model2.add(Dense(256 , activation = 'relu'))
model2.add(Dense(128 , activation = 'relu'))
model2.add(Dense(64 , activation = 'relu'))
model2.add(Dense(32 , activation = 'relu'))
model2.add(Dense(4 , activation = 'softmax'))

model2.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xception (Functional)           │ (None, None, None,     │    20,861,480 │
│                                 │ 2048)                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,429,388 (81.75 MB)

 Trainable params: 567,908 (2.17 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [13]:
model2.compile(loss = "categorical_crossentropy" , optimizer ="Adam" , metrics = ['accuracy'])

In [14]:
history2 = model2.fit(train_data ,
                      epochs = 50 ,
                      validation_data = test_data ,
          callbacks = [board("TB" , "transfer") ,
                       early() ,
                       point("transfer.ckpt")]
                      )

Epoch 1/50


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 1), output.shape=(None, 4)

In [ ]:
model2.evaluate(test_data)

In [ ]:
base.trainable = True
for i in base.layers[:-10]:
  i.trainable = False
model2.summary()

In [ ]:
model2.compile(loss = "categorical_crossentropy" , optimizer ="Adam" , metrics = ['accuracy'])

In [ ]:
history2 = model2.fit(train_data ,
                      epochs = 50 , 
                      validation_data = test_data , 
          callbacks = [board("TB" , "transfer") , early() ,
                       point("transfer.ckpt")]  ,
                      initial_epoch = history2.epoch[-1])

In [ ]:
model2.load_weights('/kaggle/working/transfer.ckpt.weights.h5')

In [ ]:
model2.evaluate(test_data)

In [ ]:
test_loss, test_acc = model2.evaluate(train_data)


In [ ]:
print(f"Test Accuracy: {test_acc}\ntest loss : {test_loss}")

In [ ]:
import numpy as np

# Get a batch of test data
test_images, test_labels = next(iter(test_data))  # Extract one batch

# Predict on the first 10 images
preds = model2.predict(test_images[:50])

# Get the predicted and actual class labels
predicted_classes = np.argmax(preds, axis=1)
actual_classes = np.argmax(test_labels[:50], axis=1)  # Assuming one-hot encoding

# Print results
for i in range(10):
    print(f"Image {i+1}: Predicted Class = {predicted_classes[i]}, Actual Class = {actual_classes[i]}")


In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

model2.save(str(ARTIFACT_DIR / 'KidneyDiseaseDetection.keras'))
model2.save(str(ARTIFACT_DIR / 'KidneyDiseaseDetection.h5'))
